In [1]:
# Cell 1 - imports and file paths
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_recall_curve, roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score, confusion_matrix, classification_report)
import joblib

FEATURES_PATH = "data/features_accounts.csv"
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)
print("Paths:", FEATURES_PATH, "->", MODEL_DIR)


Paths: data/features_accounts.csv -> model


In [2]:
# Cell 2 - load features
df = pd.read_csv(FEATURES_PATH)
print("Loaded features shape:", df.shape)
print("Label distribution:\n", df["is_mule"].value_counts())
df.head(3)


Loaded features shape: (4387, 22)
Label distribution:
 is_mule
0    4387
Name: count, dtype: int64


,account_id,is_mule,inbound_count_24h,inbound_sum_24h,inbound_unique_senders_24h,inbound_count_7d,inbound_sum_7d,inbound_unique_senders_7d,outbound_count_24h,outbound_sum_24h,...,outbound_sum_7d,outbound_unique_receivers_7d,pct_forwarded_7d,avg_forward_delay_seconds,in_degree_7d,out_degree_7d,pagerank_7d,count_in_out_ratio_7d,many_inbound_senders_24h_flag,burstiness_24h_vs_7d
0,A000000,0,0.0,0.0,0.0,1.0,2083.89,1.0,0.0,0.0,...,0.00,0.0,0.000000,-1.0,1,0,0.000214,1.000000e+09,0,0.0
1,A000001,0,0.0,0.0,0.0,1.0,705.77,1.0,0.0,0.0,...,3188.86,2.0,4.518271,85481.0,1,2,0.000177,5.000000e-01,0,0.0
2,A000002,0,0.0,0.0,0.0,1.0,70.66,1.0,0.0,0.0,...,6893.80,2.0,97.562978,-1.0,1,2,0.000242,5.000000e-01,0,0.0


In [3]:
# Cell 3 - train/test split
y = df["is_mule"].values
X = df.drop(columns=["account_id", "is_mule"])

# simple 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (3509, 20) Test: (878, 20)


In [4]:
# Cell 4 - train a baseline RandomForest with class weight to handle imbalance
rf = RandomForestClassifier(n_estimators=200, max_depth=12, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Save baseline model
joblib.dump(rf, os.path.join(MODEL_DIR, "rf_baseline.pkl"))
print("Trained RandomForest saved to", os.path.join(MODEL_DIR, "rf_baseline.pkl"))


Trained RandomForest saved to model\rf_baseline.pkl


In [6]:
# Robust training cell: stratified split -> fallback upsample -> train RandomForest -> eval -> save
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score, average_precision_score,
                             precision_recall_curve, f1_score, precision_score, recall_score)
from sklearn.utils import resample
import joblib

# Paths (adjust if different)
FEATURES_PATH = "data/features_accounts.csv"
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)

# 1) Load feature table (if not already in memory)
try:
    df
except NameError:
    df = pd.read_csv(FEATURES_PATH)

print("Loaded features shape:", df.shape)
print("Overall label distribution:\n", df["is_mule"].value_counts())

# 2) Prepare X, y
y = df["is_mule"].values
X = df.drop(columns=["account_id", "is_mule"])

# 3) Try StratifiedShuffleSplit (preferred)
sss_success = False
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

for train_idx, test_idx in sss.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    sss_success = True

print("After stratified split -> train counts:", np.unique(y_train, return_counts=True),
      " test counts:", np.unique(y_test, return_counts=True))

# 4) If training set has only one class, fallback: upsample minority in training set
unique_train = np.unique(y_train)
if len(unique_train) < 2:
    print("WARNING: training set has only one class. Applying upsampling fallback.")

    # Do a simple non-stratified split to ensure we have a test set
    X_train_tmp, X_test, y_train_tmp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    train_df = X_train_tmp.copy()
    train_df["is_mule"] = y_train_tmp

    # Separate majority/minority
    majority = train_df[train_df["is_mule"] == 0]
    minority = train_df[train_df["is_mule"] == 1]

    if len(minority) == 0:
        # Extreme fallback: synthesize a small set of positive examples by perturbing a few majority rows
        print("Extreme case: no positives in training data. Creating synthetic positives (demo-only).")
        synth = majority.sample(n=max(1, min(50, int(0.02 * len(majority)))), replace=True, random_state=42).copy()
        num_cols = synth.select_dtypes(include=[np.number]).columns.tolist()
        # small perturbation to numeric columns
        for c in num_cols:
            synth[c] = synth[c] * (1 + 0.05 * np.random.randn(len(synth)))
        synth["is_mule"] = 1
        minority = synth

    # Upsample minority to match majority size
    minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=42)
    train_balanced = pd.concat([majority, minority_upsampled]).sample(frac=1, random_state=42).reset_index(drop=True)

    X_train = train_balanced.drop(columns=["is_mule"])
    y_train = train_balanced["is_mule"].values
    print("After upsampling -> train label counts:", np.unique(y_train, return_counts=True))
else:
    print("Training set contains both classes; proceeding without upsampling.")
    # ensure variables defined in case stratified succeeded earlier
    # (X_train, y_train, X_test, y_test already set by sss.split)

# 5) Train RandomForest with class_weight='balanced' (robust for imbalanced data)
rf = RandomForestClassifier(n_estimators=200, max_depth=12, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# 6) Evaluate on test set
y_proba = rf.predict_proba(X_test)
# If predict_proba returns a single column, handle gracefully (shouldn't after fixes)
if y_proba.ndim == 1 or y_proba.shape[1] == 1:
    # fallback: use predict (0/1) as probability-like (not ideal)
    print("NOTE: model returned single-class probabilities; falling back to predict(). This indicates training labels issue.")
    y_proba_vals = rf.predict(X_test).astype(float)
else:
    y_proba_vals = y_proba[:, 1]

y_pred_default = (y_proba_vals >= 0.5).astype(int)

print("\nClassification report (threshold=0.5):")
print(classification_report(y_test, y_pred_default, digits=4))

roc = roc_auc_score(y_test, y_proba_vals) if len(np.unique(y_test))>1 else float("nan")
pr = average_precision_score(y_test, y_proba_vals) if len(np.unique(y_test))>1 else float("nan")
print(f"ROC-AUC: {roc:.4f}  PR-AUC: {pr:.4f}")

# 7) Precision-Recall curve & best F1 threshold selection
prec, rec, thr = precision_recall_curve(y_test, y_proba_vals)
# compute F1 for thresholds (align lengths)
f1_scores = (2 * prec * rec) / (prec + rec + 1e-12)
best_idx = np.nanargmax(f1_scores)
best_thr = thr[best_idx] if best_idx < len(thr) else 0.5
print("Selected threshold (best F1):", best_thr)

# 8) Feature importances
fi = pd.DataFrame({"feature": X.columns, "importance": rf.feature_importances_}).sort_values("importance", ascending=False)
fi_path = os.path.join(MODEL_DIR, "feature_importances.csv")
fi.to_csv(fi_path, index=False)
print("Saved feature importances to:", fi_path)
print("Top features:\n", fi.head(10))

# 9) Save model and simple policy (allow/otp/hold thresholds)
model_path = os.path.join(MODEL_DIR, "model.pkl")
joblib.dump(rf, model_path)
policy = {"allow_thresh": 0.3, "otp_thresh": 0.65, "chosen_thr": float(best_thr)}
with open(os.path.join(MODEL_DIR, "policy.json"), "w") as f:
    json.dump(policy, f, indent=2)

print("Saved model to:", model_path)
print("Saved policy:", policy)

# 10) Quick sample scoring helper and test
def score_features_row(row):
    """Row can be a pd.Series or dict with same columns as X"""
    x = pd.DataFrame([row])[X.columns]
    p = rf.predict_proba(x)[:,1] if rf.predict_proba(x).shape[1] > 1 else rf.predict(x).astype(float)
    prob = float(p[0])
    decision = "ALLOW" if prob < policy["allow_thresh"] else ("OTP" if prob < policy["otp_thresh"] else "HOLD")
    # simple reasons: top 3 important features with highest relative z-score for this account
    top_feats = fi.head(10)["feature"].tolist()
    acct_vals = x.iloc[0]
    reasons = []
    for f in top_feats:
        try:
            # compare to training distribution median/iqr if available in df
            med = X_train[f].median() if f in X_train.columns else 0
            if acct_vals[f] > med:
                reasons.append(f)
        except Exception:
            continue
        if len(reasons) >= 3:
            break
    return {"probability": prob, "decision": decision, "reasons": reasons}

# test on a random sample from test set
sample_row = X_test.sample(1, random_state=42).iloc[0].to_dict()
print("Sample score:", score_features_row(sample_row))


Loaded features shape: (4387, 22)
Overall label distribution:
 is_mule
0    4387
Name: count, dtype: int64
After stratified split -> train counts: (array([0]), array([3509]))  test counts: (array([0]), array([878]))
Extreme case: no positives in training data. Creating synthetic positives (demo-only).
After upsampling -> train label counts: (array([0, 1]), array([3509, 3509]))

Classification report (threshold=0.5):
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       878

    accuracy                         1.0000       878
   macro avg     1.0000    1.0000    1.0000       878
weighted avg     1.0000    1.0000    1.0000       878

ROC-AUC: nan  PR-AUC: nan
Selected threshold (best F1): 0.0
Saved feature importances to: model\feature_importances.csv
Top features:
                          feature  importance
13     avg_forward_delay_seconds    0.209311
14                  in_degree_7d    0.119702
3               inbound_count_7d    0

c:\Users\alpha\OneDrive\Desktop\Projects\AI Fraud Detection Using UPI\venv\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
